# Choice AUC Analysis — All Pairwise Region Matching with MMDDM Inclusion Criteria

This notebook computes choice selectivity (ROC-AUC) for all recorded neurons,
applies MMDDM inclusion criteria (baseline `inclusion == 1`, firing-rate,
selectivity, and trial-selection filters), then matches the `|AUC − 0.5|`
distributions separately for every pair among the 10 target brain regions:
ADS, BLA, dmFC, HPC, M1, MGB, mPFC, NAc, S1, TS.

This produces all $\binom{10}{2} = 45$ unique region pairings.


---
## 0 — Imports & settings

In [ ]:
import pickle, glob, os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

# -- Plotting defaults --
plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 12,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

# -- Consistent region colors used in every plot throughout the notebook --
REGION_COLORS = {
    'ADS':  '#d62728',
    'M1':   '#1f77b4',
    'dmFC': '#ff7f0e',
    'TS':   '#8c564b',
    'MGB':  '#9467bd',
    'HPC':  '#2ca02c',
    'NAc':  '#17becf',
    'S1':   '#e377c2',
    'mPFC': '#bcbd22',
    'BLA':  '#7f7f7f',
}

# -- Figure output directory --
FIG_DIR = os.path.join('.', 'choiceAUC_figures')
os.makedirs(FIG_DIR, exist_ok=True)
print(f'Figures will be saved to: {os.path.abspath(FIG_DIR)}')


---
## 1 — Load all session files

Each `.pkl` file contains a dictionary with three keys:
- `dfTrials`: one row per trial, with columns like `violated`, `poked_r`, `clicks_on`, `stim_dur`
- `dfCells`: one row per neuron, with columns like `unique_cellid`, `brain_region`, `raw_spike_time_s`
- `dfProbes`: probe metadata (not used here)

We wrap the load in a try/except because some pkl files may be corrupted
(e.g. truncated during copy), and we want to skip those gracefully.

In [ ]:
# ── Paths ──
PKL_DIR = '/Users/jc2988/Desktop/evapkl'
CSV_DIR = '.'

# ── Find all session pickle files ──
pkl_files = sorted(glob.glob(os.path.join(PKL_DIR, '*_CellsAndTrials.pkl')))
print(f'Found {len(pkl_files)} session file(s) in {os.path.abspath(PKL_DIR)}\n')
for f in pkl_files:
    print(f'  {os.path.basename(f)}')

# ── Load each session, skipping any that fail validation ──
sessions = []

for fpath in pkl_files:
    fname = os.path.basename(fpath)

    try:
        with open(fpath, 'rb') as fh:
            d = pickle.load(fh)
    except Exception as e:
        print(f'  ⚠ SKIPPING {fname}: could not unpickle file: {e}')
        continue

    # Validate top-level object
    if not isinstance(d, dict):
        print(f'  ⚠ SKIPPING {fname}: pickle did not contain a dict (got {type(d)})')
        continue

    # Safely get expected entries
    dfT = d.get('dfTrials', None)
    dfC = d.get('dfCells', None)

    # Require trials table
    if dfT is None:
        print(f'  ⚠ SKIPPING {fname}: dfTrials is None or missing')
        continue

    # Require cells table
    if dfC is None:
        print(f'  ⚠ SKIPPING {fname}: dfCells is None or missing')
        continue

    # Optional: make sure these are actually DataFrames
    if not isinstance(dfT, pd.DataFrame):
        print(f'  ⚠ SKIPPING {fname}: dfTrials is not a DataFrame (got {type(dfT)})')
        continue

    if not isinstance(dfC, pd.DataFrame):
        print(f'  ⚠ SKIPPING {fname}: dfCells is not a DataFrame (got {type(dfC)})')
        continue

    # Some sessions store brain_region as an index level rather than a column.
    # Normalize it to always be a regular column for consistent downstream access.
    if 'brain_region' not in dfC.columns:
        index_names = list(dfC.index.names) if dfC.index.names is not None else []
        if 'brain_region' in index_names:
            dfC = dfC.reset_index(level='brain_region')

    sessions.append({
        'source_file': fname,
        'dfTrials': dfT,
        'dfCells': dfC,
    })

# Store all sessions in a DataFrame for easy iteration
combined = pd.DataFrame(sessions)
print(f'\nLoaded {len(combined)} valid session(s).')

---
## 2 — Repair missing brain-region assignments

Many sessions are recorded with Neuropixels probes but ship without
brain-region labels in `dfCells`. We repair these using
`uberphys_brain_regions.csv`, which maps:

    (rat, probe serial number, electrode site range) → anatomical region

For each neuron that is missing a region label, we look up its rat, probe
serial, and electrode number in this CSV and assign the matching region.
Multi-shank probes require an additional `bank`/`shank` field to disambiguate.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Load the electrode-to-region mapping CSV
# ══════════════════════════════════════════════════════════════════
# This CSV (uberphys_brain_regions.csv) contains one row per brain-region
# boundary per probe. Columns include:
#   rat          - rat name (e.g. 'A324')
#   serial       - unique serial number of the Neuropixels probe
#   shank        - shank index for multi-shank probes (NaN for single-shank)
#   min_electrode, max_electrode - electrode site range for this region
#   region       - anatomical region name (e.g. 'M1', 'ADS', 'M2', etc.)

region_csv_path = os.path.join(CSV_DIR, 'uberphys_brain_regions.csv')
region_map_df = pd.read_csv(region_csv_path, encoding='utf-8-sig')

# Clean up the region column: strip whitespace, convert literal 'nan' strings to NaN
region_map_df['region'] = region_map_df['region'].astype(str).str.strip()
region_map_df.loc[region_map_df['region'] == 'nan', 'region'] = np.nan

# Ensure serial is a string (some may be numeric in the CSV)
region_map_df['serial'] = region_map_df['serial'].astype(str)

# Convert max_electrode to numeric; replace infinity with a large sentinel
# value so that open-ended ranges (e.g. "electrode >= 300") work correctly
region_map_df['max_electrode'] = pd.to_numeric(
    region_map_df['max_electrode'], errors='coerce'
).replace(np.inf, 99999)

print(f'Loaded region map: {len(region_map_df)} rows, '
      f'{region_map_df["serial"].nunique()} unique probes')


def assign_brain_regions(dfCells, region_map_df):
    """
    For neurons missing a brain_region label, look up the region from the
    electrode-to-region mapping CSV based on (rat, probe serial, electrode site).

    Parameters
    ----------
    dfCells : pd.DataFrame
        Neuron-level data. May or may not already have a 'brain_region' column.
        If it exists, only rows with missing/invalid values are repaired.
    region_map_df : pd.DataFrame
        The electrode-to-region mapping table loaded from uberphys_brain_regions.csv.

    Returns
    -------
    dfCells : pd.DataFrame (modified in place)
    n_assigned : int — number of neurons successfully assigned a region
    n_failed : int — number of neurons where lookup failed
    """
    # ── Ensure brain_region column exists ──
    # It might be stored as an index level instead of a column, or missing entirely.
    if 'brain_region' not in dfCells.columns:
        if 'brain_region' in dfCells.index.names:
            dfCells = dfCells.reset_index(level='brain_region')
        else:
            dfCells['brain_region'] = np.nan

    # ── Identify neurons that need a region assignment ──
    # A neuron "needs" a region if its brain_region is NaN, empty string,
    # or one of the placeholder values like 'unknown', 'none', 'unassigned'.
    needs_region = (
        dfCells['brain_region'].isna() |
        (dfCells['brain_region'].astype(str).str.strip() == '') |
        (dfCells['brain_region'].astype(str).str.lower().isin(
            ['unknown', 'none', 'nan', 'unassigned']))
    )
    if needs_region.sum() == 0:
        return dfCells, 0, 0

    # ── Extract rat names from the index if available ──
    # The dfCells index structure varies across sessions; sometimes 'ratname'
    # is an index level, sometimes it's not present at all.
    if 'ratname' in dfCells.index.names:
        rat_values = dfCells.index.get_level_values('ratname')
    else:
        rat_values = pd.Series(['unknown'] * len(dfCells), index=dfCells.index)

    # ── Identify multi-shank probes ──
    # Multi-shank probes have a 'shank' value in the region map. For these,
    # we need to additionally match on the neuron's 'bank' field to get
    # the correct shank assignment.
    multi_shank_serials = set(
        region_map_df.loc[region_map_df['shank'].notna(), 'serial'].unique()
    )

    n_assigned = 0
    n_failed = 0

    # ── Loop over each neuron that needs a region ──
    for idx in dfCells.index[needs_region]:
        row = dfCells.loc[idx]

        # Extract rat name and probe serial from the index.
        # The index might be a tuple (ratname, date, serial, ...) or a flat integer.
        if isinstance(idx, tuple):
            rat, serial = str(idx[0]), str(idx[2])
        else:
            rat = str(rat_values[idx]) if idx in rat_values.index else 'unknown'
            serial = 'unknown'

        # ── Get the neuron's electrode site number ──
        # Try 'electrode' first, fall back to 'distance_from_tip' if absent.
        electrode = row.get('electrode', np.nan)
        if pd.isna(electrode):
            electrode = row.get('distance_from_tip', np.nan)
        if pd.isna(electrode):
            # Can't look up region without knowing electrode position
            n_failed += 1
            continue
        electrode = float(electrode)

        # ── Filter the region map to this specific probe ──
        probe_map = region_map_df[
            (region_map_df['rat'] == rat) & (region_map_df['serial'] == serial)
        ]
        if len(probe_map) == 0:
            # No mapping exists for this rat/probe combination
            n_failed += 1
            continue

        # ── For multi-shank probes, further filter by shank/bank ──
        if serial in multi_shank_serials:
            bank = row.get('bank', np.nan)
            if pd.notna(bank):
                probe_map = probe_map[probe_map['shank'] == float(bank)]

        # ── Find the region whose electrode range contains this neuron ──
        match = probe_map[
            (probe_map['min_electrode'] <= electrode) &
            (probe_map['max_electrode'] >= electrode) &
            (probe_map['region'].notna())
        ]
        if len(match) >= 1:
            # Take the first matching region (there should be at most one)
            dfCells.at[idx, 'brain_region'] = match.iloc[0]['region']
            n_assigned += 1
        else:
            # Electrode site fell outside all defined region boundaries
            n_failed += 1

    return dfCells, n_assigned, n_failed


# ══════════════════════════════════════════════════════════════════
# Apply brain-region repair to every loaded session
# ══════════════════════════════════════════════════════════════════
print('\nRepairing brain-region assignments:')
for i, row in combined.iterrows():
    src = row['source_file']
    dfC = row['dfCells']

    # Count how many neurons currently lack a valid brain_region
    if 'brain_region' in dfC.columns:
        n_missing = (
            dfC['brain_region'].isna() |
            (dfC['brain_region'].astype(str).str.strip() == '') |
            (dfC['brain_region'].astype(str).str.lower().isin(
                ['unknown', 'none', 'nan', 'unassigned']))
        ).sum()
    else:
        n_missing = len(dfC)

    if n_missing == 0:
        print(f'  {src}: all {len(dfC)} neurons already have region labels')
        continue

    # Run the repair function and update the session's dfCells in place
    dfC_fixed, n_a, n_f = assign_brain_regions(dfC, region_map_df)
    combined.at[i, 'dfCells'] = dfC_fixed
    print(f'  {src}: {n_missing} needed repair — {n_a} assigned, {n_f} unresolved')

print('Done.')


---
## 3 — MMDDM inclusion criteria

After computing choice AUC for all neurons, we apply the MMDDM inclusion criteria:
1. **Firing rate ≥ 2 spikes/s** — removes low-firing units unlikely to carry reliable information.
2. **|AUC − 0.5| > 0.03125** (= 2⁻⁵) — retains only neurons with measurable choice selectivity.
3. **Trial selection** — non-violated, fixation held, response time ≤ 5 s, stimulus duration ≥ 0.5 s.
4. **Region exclusion** — excludes CC and unknown.


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MMDDM inclusion criteria (derived from SpkTr pipeline)
# ══════════════════════════════════════════════════════════════════
# These match the MATLAB pipeline exactly:
#   process_Cells_file.m:  FR = numel(raw_spike_time_s) / fileTimeSecs
#                          neuronindices = firing_rate >= min_firing_rate
#   data_for_FHMDDM_models.m:  sel = |AUC - 0.5| > 2^-5
#
# Trial selection (findtrials.m + data_for_FHMDDM_models.m):
#   ~violated
#   cpoke_out > cpoke_req_end  (animal held fixation through required period)
#   response_time <= 5 s       (spoke - cpoke_req_end <= max_response_time_s)
#   ntimesteps >= 50           (stim_dur >= 0.5 s)

MIN_FIRING_RATE     = 2.0        # spikes per second (settings_2024_05_30)
MIN_AUC_DEV         = 2**-5      # = 0.03125 exactly (data_for_FHMDDM_models.m)
MAX_RESPONSE_TIME_S = 5.0        # seconds (settings.max_response_time_s)
WINDOW_DUR          = 0.500      # 500 ms spike-counting window for AUC
EXCLUDED_REGIONS    = ['CC', 'unknown']  # excluded in data_for_FHMDDM_models.m

print(f"MMDDM inclusion criteria:")
print(f"  Firing rate \u2265 {MIN_FIRING_RATE} sp/s  (total spikes / recording duration)")
print(f"  |AUC - 0.5| > {MIN_AUC_DEV}  (= 2\u207b\u2075)")
print(f"  Max response time \u2264 {MAX_RESPONSE_TIME_S} s")
print(f"  Stimulus duration \u2265 {WINDOW_DUR} s  (ntimesteps \u2265 50)")
print(f"  Excluded brain regions: {EXCLUDED_REGIONS}")

---
## 4 — Compute choice AUC for all neurons

We compute choice selectivity using spike counts during the first 0.5 s from stimulus onset on each trial (excluding trials ending before 0.5 s after stimulus onset). Firing rate is also computed per neuron for later filtering.

In [ ]:
def compute_choice_auc_session(dfTrials, dfCells, restrict_cellids=None):
    """
    Compute choice AUC for neurons in one recording session using MMDDM
    inclusion criteria (derived from the SpkTr MATLAB pipeline).

    Trial selection matches findtrials.m + data_for_FHMDDM_models.m:
      - ~violated
      - cpoke_out > cpoke_req_end  (held fixation through required period)
      - response_time <= MAX_RESPONSE_TIME_S
      - stim_dur >= WINDOW_DUR  (ntimesteps >= 50 at 10 ms bins)

    Firing rate matches process_Cells_file.m:
      FR = numel(raw_spike_time_s) / fileTimeSecs

    AUC is computed on the first 500 ms from stimulus onset (clicks_on),
    matching sum(spiketrain(n, 1:50)) in MATLAB.
    """
    # ── Step 0: Restrict to neurons flagged for inclusion ──
    # This mirrors apply_inclusion_criteria in process_Cells_file.m
    if 'inclusion' in dfCells.columns:
        dfCells = dfCells[dfCells['inclusion'] == 1].copy()

    # ── Step 1: Filter trials to match findtrials.m ──
    # findtrials.m:
    #   response_time_s = spoke_s - cpoke_req_end_s
    #   ~violated & cpoke_out > cpoke_req_end & response_time <= max_response_time_s
    valid = dfTrials.copy()

    # Filter: not violated
    if 'violated' in valid.columns:
        valid = valid[valid['violated'] == 0]

    # Filter: cpoke_out > cpoke_req_end (animal held fixation through required period)
    # This implicitly excludes never_cpoked trials
    if 'cpoke_out' in valid.columns and 'cpoke_req_end' in valid.columns:
        valid = valid[valid['cpoke_out'] > valid['cpoke_req_end']]
    elif 'never_cpoked' in valid.columns:
        # Fallback: if cpoke_out/cpoke_req_end not available, use never_cpoked
        valid = valid[valid['never_cpoked'] == 0]

    # Filter: response_time <= MAX_RESPONSE_TIME_S
    if 'spoke' in valid.columns and 'cpoke_req_end' in valid.columns:
        response_time = valid['spoke'] - valid['cpoke_req_end']
        valid = valid[response_time <= MAX_RESPONSE_TIME_S]

    # Filter: valid choice
    if 'poked_r' in valid.columns:
        valid = valid[valid['poked_r'].notna()]

    # Filter: valid stimulus timing
    if 'clicks_on' in valid.columns:
        valid = valid[valid['clicks_on'].notna()]

    # Filter: stim_dur >= WINDOW_DUR (matches ntimesteps >= 50 in MATLAB)
    if 'stim_dur' in valid.columns:
        valid = valid[valid['stim_dur'] >= WINDOW_DUR]

    valid = valid.copy()

    if len(valid) < 10:
        return [], None

    # ── Step 2: Spike-count window ──
    valid['win_start_abs'] = valid['clicks_on']
    valid['win_end_abs'] = valid['clicks_on'] + WINDOW_DUR
    valid['choice'] = valid['poked_r'].astype(int)

    if valid['choice'].nunique() < 2:
        return [], None

    win_starts = valid['win_start_abs'].values
    win_ends   = valid['win_end_abs'].values
    choices    = valid['choice'].values

    # ── Step 3: Estimate recording duration for FR computation ──
    # MMDDM computes FR = numel(raw_spike_time_s) / Cells.rec.ap_meta.fileTimeSecs
    # We try to recover fileTimeSecs from the dfCells metadata.
    # Fallback: use max spike time across all neurons (lower bound on duration).
    recording_duration = None
    if 'recording_duration_s' in dfCells.columns:
        recording_duration = dfCells['recording_duration_s'].iloc[0]
    elif 'fileTimeSecs' in dfCells.columns:
        recording_duration = dfCells['fileTimeSecs'].iloc[0]

    if recording_duration is None:
        # Fallback: estimate from the latest spike time across all neurons
        max_spike_time = 0.0
        for _, cell_row in dfCells.iterrows():
            st = cell_row.get('raw_spike_time_s', None)
            if st is not None:
                arr = np.asarray(st).ravel()
                if arr.size > 0:
                    max_spike_time = max(max_spike_time, float(arr[-1] if np.all(np.diff(arr) >= 0) else arr.max()))
        recording_duration = max_spike_time if max_spike_time > 0 else None

    # ── Step 4: Loop over neurons ──
    results = []
    for cell_idx, cell_row in dfCells.iterrows():
        uid = cell_row.get('unique_cellid', np.nan)
        if pd.isna(uid):
            continue
        if restrict_cellids is not None and uid not in restrict_cellids:
            continue

        spike_times = np.asarray(cell_row['raw_spike_time_s']).ravel()
        if spike_times.size < 2:
            continue
        if not np.all(np.diff(spike_times) >= 0):
            spike_times = np.sort(spike_times)

        # ── Firing rate: MMDDM method ──
        # FR = total_spikes / recording_duration (not spike_span!)
        if recording_duration is not None and recording_duration > 0:
            firing_rate = spike_times.size / recording_duration
        else:
            # Last-resort fallback (will be flagged in verification)
            spike_span = spike_times[-1] - spike_times[0]
            firing_rate = spike_times.size / spike_span if spike_span > 0 else 0.0

        # ── Spike counts per trial ──
        spike_counts = np.empty(len(valid), dtype=int)
        for t in range(len(valid)):
            spike_counts[t] = (
                np.searchsorted(spike_times, win_ends[t], side='left') -
                np.searchsorted(spike_times, win_starts[t], side='left')
            )

        # ── ROC-AUC ──
        if len(np.unique(spike_counts)) < 2:
            auc_val = 0.5
        else:
            auc_val = roc_auc_score(choices, spike_counts)

        # ── Brain region ──
        br_raw = cell_row.get('brain_region', np.nan)
        if pd.isna(br_raw) or str(br_raw).strip() in ('', 'nan', 'None'):
            brain_region = 'unknown'
        else:
            brain_region = str(br_raw).strip()

        results.append({
            'unique_cellid':     uid,
            'brain_region':      brain_region,
            'auc':               auc_val,
            'abs_auc_deviation': abs(auc_val - 0.5),
            'firing_rate':       firing_rate,
            'n_trials_used':     len(valid),
            'n_total_spikes':    int(spike_times.size),
            'cell_index':        cell_idx,
        })

    return results, recording_duration

In [ ]:
# ── Compute AUC for included neurons in every session ──
all_results_full = []
session_rec_durations = {}

for _, session_row in combined.iterrows():
    dfT = session_row['dfTrials']
    dfC = session_row['dfCells']
    src = session_row['source_file']

    print(f'{src} ({len(dfC)} total neurons) ...', end=' ')
    session_results, rec_dur = compute_choice_auc_session(dfT, dfC, restrict_cellids=None)

    session_rec_durations[src] = rec_dur
    for r in session_results:
        r['source_file'] = src
    all_results_full.extend(session_results)

    fr_source = 'fileTimeSecs' if rec_dur is not None else 'spike_span (fallback)'
    print(f'{len(session_results)} AUCs computed  '
          f'(rec_dur={rec_dur:.1f}s, FR via {fr_source})' if rec_dur else
          f'{len(session_results)} AUCs computed  (FR via spike_span fallback)')

auc_df_full = pd.DataFrame(all_results_full)
print(f'\nTotal neurons with AUC: {len(auc_df_full)}')
print(f'\n|AUC - 0.5| summary:')
print(auc_df_full['abs_auc_deviation'].describe())
print(f'\nFiring rate summary (MMDDM method):')
print(auc_df_full['firing_rate'].describe())

---
## 4b — Apply MMDDM inclusion criteria

Remove neurons with firing rate < 2 sp/s or |AUC − 0.5| ≤ 0.03125.


In [ ]:
# ══════════════════════════════════════════════════════════════════
# Apply MMDDM inclusion criteria: FR >= 2 sp/s AND |AUC-0.5| > 2^-5
# Also exclude CC and unknown brain regions (data_for_FHMDDM_models.m)
# ══════════════════════════════════════════════════════════════════
n_before = len(auc_df_full)

# Filter 1: exclude CC and unknown brain regions
region_mask = ~auc_df_full['brain_region'].isin(EXCLUDED_REGIONS)
n_fail_region = (~region_mask).sum()

# Filter 2: firing rate >= MIN_FIRING_RATE (2.0 sp/s)
fr_mask = auc_df_full['firing_rate'] >= MIN_FIRING_RATE
n_fail_fr = (~fr_mask).sum()

# Filter 3: |AUC - 0.5| > MIN_AUC_DEV (2^-5 = 0.03125)
auc_mask = auc_df_full['abs_auc_deviation'] > MIN_AUC_DEV
n_fail_auc = (~auc_mask).sum()

# Apply all three
combined_mask = region_mask & fr_mask & auc_mask
auc_df_filtered = auc_df_full[combined_mask].copy()

print(f"MMDDM inclusion criteria applied:")
print(f"  Full population:                          {n_before}")
print(f"  Removed (region in {EXCLUDED_REGIONS}):  {n_fail_region}")
print(f"  Removed (FR < {MIN_FIRING_RATE} sp/s):                  {(~fr_mask).sum()}")
print(f"  Removed (|AUC-0.5| <= 2^-5 = {MIN_AUC_DEV}):     {(~auc_mask).sum()}")
print(f"  Removed (any criterion):                  {n_before - len(auc_df_filtered)}")
print(f"  Retained:                                 {len(auc_df_filtered)} "
      f"({len(auc_df_filtered)/n_before:.1%})")
print(f"\n|AUC - 0.5| summary (MMDDM-filtered):")
print(auc_df_filtered['abs_auc_deviation'].describe())
print(f"\nFiring rate summary (MMDDM-filtered):")
print(auc_df_filtered['firing_rate'].describe())

---
## 5 — Region grouping

Map fine-grained anatomical labels to grouped regions
(`M2` + `Cg1` → `dmFC`, `MO` + `PrL` → `mPFC`,
`DGd` + `DGv` + `DG` + `CA1` → `HPC`), then enumerate all
$\binom{10}{2} = 45$ unique pairings among the 10 target regions.


In [ ]:
from itertools import combinations

# -- Region grouping map --
REGION_MAP = {
    'M2':  'dmFC',
    'Cg1': 'dmFC',
    'MO':  'mPFC',
    'PrL': 'mPFC',
    'DGd': 'HPC',
    'DGv': 'HPC',
    'DG':  'HPC',
    'CA1': 'HPC',
}

# All 10 target regions and all C(10,2) = 45 unique pairings.
TARGET_REGIONS = sorted(['ADS', 'BLA', 'dmFC', 'HPC', 'M1', 'MGB', 'mPFC', 'NAc', 'S1', 'TS'])
REGION_PAIRS = list(combinations(TARGET_REGIONS, 2))

# -- Apply region grouping to the MMDDM-filtered neuron table --
auc_filtered_regions = auc_df_filtered.dropna(subset=['auc']).copy()
auc_filtered_regions['region_group'] = (
    auc_filtered_regions['brain_region']
    .map(REGION_MAP)
    .fillna(auc_filtered_regions['brain_region'])
)

# Keep only neurons in the target regions.
auc_filtered_regions = auc_filtered_regions[
    auc_filtered_regions['region_group'].isin(TARGET_REGIONS)
].copy()

print(f'MMDDM-filtered neurons in target regions: {len(auc_filtered_regions)}')
print(auc_filtered_regions.groupby('region_group').size().sort_index().to_string())
print()
print(f'Number of region pairs to match: {len(REGION_PAIRS)}')
print('Region pairs:')
for i, (reg_a, reg_b) in enumerate(REGION_PAIRS):
    print(f'  {i+1:2d}. {reg_a}–{reg_b}')


---
## 6 — Match |AUC − 0.5| distributions for each region pair

For each of the 45 region pairs, matching is done independently within each
recording session. The algorithm searches over low/high trims of each region's
sorted |AUC − 0.5| distribution to find the configuration that retains the
most neurons while satisfying mean, max, and size tolerances.


In [ ]:
# ================================================================
# Matching parameters
# ================================================================
# These tolerances define when two trimmed |AUC-0.5| distributions
# are considered a successful match.
MEAN_TOL = 0.005      # max allowed difference in mean |AUC-0.5|
MAX_TOL = 0.005       # max allowed difference in max  |AUC-0.5|
SIZE_TOL = 10         # max allowed difference in region population size
MAX_TRIM_FRAC = 0.5   # allow trimming up to half of each region

# Bin edges reused in the distribution plots below.
BIN_WIDTH = 0.02
BIN_EDGES = np.arange(0, 0.5 + BIN_WIDTH, BIN_WIDTH)

print(f'Mean tolerance:  {MEAN_TOL}')
print(f'Max  tolerance:  {MAX_TOL}')
print(f'Size tolerance:  {SIZE_TOL}')
print(f'Max trim frac:   {MAX_TRIM_FRAC}')


def _find_best_trim_for_region(
    sorted_arr,
    prefix_sum,
    target_mean,
    target_max,
    mean_tol,
    max_tol,
    target_size=None,
    size_tol=None,
):
    '''
    For one region's sorted |AUC-0.5| array, find the (lo, hi) trim
    that matches the target mean and max while keeping the most neurons.
    '''
    n = len(sorted_arr)
    if n == 0:
        return None

    # Restrict the top of the retained distribution so its maximum
    # can fall within the requested max-tolerance window.
    max_lower = target_max - max_tol
    max_upper = target_max + max_tol

    idx_lo = int(np.searchsorted(sorted_arr, max_lower, side='left'))
    idx_hi = int(np.searchsorted(sorted_arr, max_upper, side='right')) - 1

    hi_end = min(n - 1 - idx_lo, n - 1)
    hi_start = max(n - 1 - idx_hi, 0)
    if hi_start > hi_end:
        return None

    # Optionally require the retained population size to be close to
    # the reference region's retained size.
    if target_size is not None and size_tol is not None:
        min_kept = max(1, target_size - size_tol)
        max_kept = target_size + size_tol
    else:
        min_kept = 1
        max_kept = n

    # Allow trimming from the bottom only up to MAX_TRIM_FRAC.
    max_lo = min(n - 1, max(1, int(n * MAX_TRIM_FRAC)))
    best = None
    best_trim_total = n + 1

    for lo in range(0, max_lo + 1):
        for hi in range(hi_start, hi_end + 1):
            kept = n - lo - hi
            if kept < min_kept or kept > max_kept:
                continue
            if lo + hi >= best_trim_total:
                continue

            upper = n - hi
            sub_mean = (prefix_sum[upper] - prefix_sum[lo]) / kept
            if abs(sub_mean - target_mean) <= mean_tol:
                best = (lo, hi)
                best_trim_total = lo + hi
                break

    return best


def match_abs_auc_pairwise(
    auc_df,
    region_pair,
    session_col='source_file',
    region_col='region_group',
    match_col='abs_auc_deviation',
    id_col='unique_cellid',
    mean_tol=MEAN_TOL,
    max_tol=MAX_TOL,
    size_tol=SIZE_TOL,
):
    '''
    Match |AUC-0.5| distributions for a single region pair by optimal trimming.

    Matching is done independently within each session:
      1. Sort each region's |AUC-0.5| values.
      2. Try both regions as the reference distribution.
      3. Search over low/high trims of the reference region.
      4. Find the best matching trim in the other region.
      5. Keep the best valid configuration that retains the most neurons.
    '''
    reg_a, reg_b = region_pair
    matched_ids = {}
    keep_indices = []
    diagnostics = {}

    for session, sess_group in auc_df.groupby(session_col):
        matched_ids[session] = {}

        # Skip sessions that do not contain both members of the pair.
        regions_present = set(sess_group[region_col].unique())
        if reg_a not in regions_present or reg_b not in regions_present:
            missing_reg = reg_a if reg_a not in regions_present else reg_b
            print(f'  {session}: skipping -- missing {missing_reg}')
            continue

        region_dfs = {}
        sorted_arrays = {}
        prefix_sums = {}

        # Pre-sort each region and cache prefix sums so the mean of
        # any retained subrange can be computed quickly.
        for reg in region_pair:
            reg_df = sess_group[sess_group[region_col] == reg].copy()
            order = reg_df[match_col].argsort()
            reg_df = reg_df.iloc[order].reset_index(drop=False)
            reg_df = reg_df.rename(columns={'index': 'orig_index'})
            arr = reg_df[match_col].values

            region_dfs[reg] = reg_df
            sorted_arrays[reg] = arr
            prefix = np.zeros(len(arr) + 1)
            prefix[1:] = np.cumsum(arr)
            prefix_sums[reg] = prefix

        best_total = -1
        best_config = None

        # Let each region serve as the reference in turn.
        for ref_reg in region_pair:
            other_reg = reg_b if ref_reg == reg_a else reg_a
            ref_arr = sorted_arrays[ref_reg]
            ref_ps = prefix_sums[ref_reg]
            n_ref = len(ref_arr)
            max_ref_trim = min(n_ref - 1, max(1, int(n_ref * MAX_TRIM_FRAC)))

            for lo_ref in range(max_ref_trim + 1):
                for hi_ref in range(max_ref_trim + 1):
                    kept_ref = n_ref - lo_ref - hi_ref
                    if kept_ref < 1:
                        continue

                    upper_ref = n_ref - hi_ref
                    ref_mean = (ref_ps[upper_ref] - ref_ps[lo_ref]) / kept_ref
                    ref_max = ref_arr[upper_ref - 1]

                    # Match the non-reference region to the current
                    # trimmed reference summary statistics.
                    other_trim = _find_best_trim_for_region(
                        sorted_arrays[other_reg],
                        prefix_sums[other_reg],
                        ref_mean,
                        ref_max,
                        mean_tol,
                        max_tol,
                        target_size=kept_ref,
                        size_tol=size_tol,
                    )
                    if other_trim is None:
                        continue

                    config = {
                        ref_reg: (lo_ref, hi_ref),
                        other_reg: other_trim,
                    }

                    kept_means = []
                    kept_maxs = []
                    kept_sizes = []
                    total = 0

                    for reg in region_pair:
                        lo, hi = config[reg]
                        arr = sorted_arrays[reg]
                        upper = len(arr) - hi
                        sub = arr[lo:upper]
                        if len(sub) == 0:
                            total = -1
                            break
                        kept_means.append(sub.mean())
                        kept_maxs.append(sub[-1])
                        kept_sizes.append(len(sub))
                        total += len(sub)

                    if total < 0:
                        continue

                    # Enforce the final pairwise constraints exactly.
                    if abs(kept_means[0] - kept_means[1]) > mean_tol:
                        continue
                    if abs(kept_maxs[0] - kept_maxs[1]) > max_tol:
                        continue
                    if abs(kept_sizes[0] - kept_sizes[1]) > size_tol:
                        continue

                    if total > best_total:
                        best_total = total
                        best_config = config

        if best_config is None:
            print(f'  {session}: no feasible trim found')
            continue

        means_final = []
        maxs_final = []
        sizes_final = []
        session_total = 0

        # Record the kept neuron IDs and the final matched summary
        # statistics for this session.
        for reg in region_pair:
            lo, hi = best_config[reg]
            reg_df = region_dfs[reg]
            arr = sorted_arrays[reg]
            upper = len(arr) - hi
            kept = reg_df.iloc[lo:upper]
            matched_ids[session][reg] = kept[id_col].values
            keep_indices.extend(kept['orig_index'].tolist())

            sub = arr[lo:upper]
            means_final.append(sub.mean())
            maxs_final.append(sub[-1])
            sizes_final.append(len(sub))
            session_total += len(sub)

        pre_total = sum(len(sorted_arrays[reg]) for reg in region_pair)
        pct = 100 * session_total / pre_total if pre_total > 0 else 0
        print(f'  {session}: retained {session_total}/{pre_total} neurons ({pct:.1f}%)')
        for reg in region_pair:
            lo, hi = best_config[reg]
            n_orig = len(sorted_arrays[reg])
            n_kept = len(matched_ids[session][reg])
            arr = sorted_arrays[reg]
            upper = len(arr) - hi
            sub = arr[lo:upper]
            print(
                f'    {reg:>5s}: {n_kept}/{n_orig}  '
                f'(trim lo={lo}, hi={hi})  '
                f'mean|AUC-0.5|={sub.mean():.4f}  '
                f'max={sub[-1]:.4f}'
            )

        diagnostics[session] = {
            'pair': f'{reg_a}-{reg_b}',
            'regions': list(region_pair),
            'config': best_config,
            'means': np.asarray(means_final),
            'maxs': np.asarray(maxs_final),
            'sizes': np.asarray(sizes_final, dtype=int),
            'pre_total': pre_total,
            'post_total': session_total,
        }

    matched_df = auc_df.loc[keep_indices].copy()
    return matched_ids, matched_df, diagnostics


In [ ]:
pairwise_results = {}
pairwise_summary_rows = []

# Run the same matching procedure independently for every requested pair.
for reg_a, reg_b in REGION_PAIRS:
    pair_label = f'{reg_a}-{reg_b}'
    pair_input = auc_filtered_regions[
        auc_filtered_regions['region_group'].isin([reg_a, reg_b])
    ].copy()

    print('=' * 80)
    print(f'PAIR: {pair_label}')
    print('=' * 80)
    print(f'Neurons entering matching: {len(pair_input)}')
    print(pair_input.groupby('region_group').size().sort_index().to_string())
    print()
    print('Matching |AUC - 0.5| distributions (within-session optimal trim):')

    matched_ids, matched_df_pair, diagnostics = match_abs_auc_pairwise(
        pair_input,
        (reg_a, reg_b),
    )

    # Store all outputs for this pair so later cells can reuse them for
    # pooled sanity checks, per-session plots, and CSV export.
    pairwise_results[pair_label] = {
        'pair': (reg_a, reg_b),
        'input_df': pair_input,
        'matched_ids': matched_ids,
        'matched_df': matched_df_pair,
        'diagnostics': diagnostics,
    }

    matched_counts = matched_df_pair['region_group'].value_counts().to_dict()
    matched_sessions = [session for session, ids in matched_ids.items() if ids]

    print()
    print(f'Total matched neurons: {len(matched_df_pair)}')
    print(f'Matched sessions: {len(matched_sessions)}')
    print(f'Matched neurons in {reg_a}: {matched_counts.get(reg_a, 0)}')
    print(f'Matched neurons in {reg_b}: {matched_counts.get(reg_b, 0)}')

    print('\n-- Matching constraint check (per session) --')
    if not diagnostics:
        print('  No sessions matched for this pair.')
    for session, diag in diagnostics.items():
        sess_label = session.replace('_CellsAndTrials.pkl', '')
        mean_spread = float(np.abs(np.diff(diag['means']))[0])
        max_spread = float(np.abs(np.diff(diag['maxs']))[0])
        size_spread = int(np.abs(np.diff(diag['sizes']))[0])
        print(f'  {sess_label}:')
        print(f'    mean |AUC-0.5| spread: {mean_spread:.6f}  (tol = {MEAN_TOL})')
        print(f'    max  |AUC-0.5| spread: {max_spread:.6f}  (tol = {MAX_TOL})')
        print(f'    size spread:           {size_spread}  (tol = {SIZE_TOL})')

    # Save one summary row per pair for a compact notebook-level overview.
    pairwise_summary_rows.append({
        'pair': pair_label,
        'region_A': reg_a,
        'region_B': reg_b,
        'n_A_input': int((pair_input['region_group'] == reg_a).sum()),
        'n_B_input': int((pair_input['region_group'] == reg_b).sum()),
        'n_A_matched': int(matched_counts.get(reg_a, 0)),
        'n_B_matched': int(matched_counts.get(reg_b, 0)),
        'n_sessions_matched': len(matched_sessions),
        'n_total_matched': len(matched_df_pair),
    })
    print()

pairwise_summary = pd.DataFrame(pairwise_summary_rows)
print('=' * 80)
print('PAIRWISE MATCH SUMMARY')
print('=' * 80)
print(pairwise_summary.to_string(index=False))


---
## 7 — Matched |AUC − 0.5| distributions: before vs. after matching

For each of the 45 region pairs, overlay the pooled |AUC − 0.5|
distributions before and after the within-session optimal trim.


In [ ]:
n_pairs = len(REGION_PAIRS)
n_cols = 5
n_rows = int(np.ceil(n_pairs / n_cols))

# ── Figure: post-matching pooled distributions for all 45 pairs ──
fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(3.8 * n_cols, 3.0 * n_rows),
    sharex=True,
)
axes_flat = axes.ravel()

for idx, (reg_a, reg_b) in enumerate(REGION_PAIRS):
    ax = axes_flat[idx]
    pair_label = f'{reg_a}-{reg_b}'
    result = pairwise_results[pair_label]
    matched_df_pair = result['matched_df']

    vals_a = matched_df_pair.loc[
        matched_df_pair['region_group'] == reg_a, 'abs_auc_deviation'
    ].values
    vals_b = matched_df_pair.loc[
        matched_df_pair['region_group'] == reg_b, 'abs_auc_deviation'
    ].values

    if len(vals_a) == 0 and len(vals_b) == 0:
        ax.text(0.5, 0.5, 'no data', transform=ax.transAxes,
                ha='center', va='center', color='gray', fontsize=8)
    else:
        ax.hist(vals_a, bins=BIN_EDGES, color=REGION_COLORS[reg_a],
                alpha=0.55, edgecolor='white', linewidth=0.3,
                label=f'{reg_a} ({len(vals_a)})')
        ax.hist(vals_b, bins=BIN_EDGES, histtype='step',
                color=REGION_COLORS[reg_b], linewidth=1.5,
                label=f'{reg_b} ({len(vals_b)})')
        ax.legend(fontsize=6, frameon=False, loc='upper right')

    ax.set_xlim(0, 0.5)
    ax.set_title(pair_label, fontsize=8, fontweight='bold')
    if idx % n_cols == 0:
        ax.set_ylabel('Count', fontsize=8)

# Hide unused subplots.
for idx in range(n_pairs, len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.suptitle(
    'Matched |AUC − 0.5| distributions (all 45 pairings)',
    fontsize=13, fontweight='bold',
)
fig.supxlabel('|AUC − 0.5|', fontsize=11)
plt.tight_layout()
plt.savefig(
    os.path.join(FIG_DIR, 'fig_matched_auc_distributions_all_pairs.png'),
    dpi=200, bbox_inches='tight',
)
plt.show()


---
## 8 — Export matched neuron IDs and AUCs to CSV

Save one combined CSV containing all pairwise matches, and one CSV per
pairing so each matched comparison can be inspected separately.


In [ ]:
rows = []
pair_csv_dir = 'matched_pairwise_csvs'
os.makedirs(pair_csv_dir, exist_ok=True)

# Export both:
#   1. one combined CSV across all pairs
#   2. one CSV per pair for easy downstream use
for pair_label, result in pairwise_results.items():
    reg_a, reg_b = result['pair']
    matched_ids = result['matched_ids']
    matched_df_pair = result['matched_df']
    pair_rows = []

    for session in sorted(matched_ids.keys()):
        regions_dict = matched_ids[session]
        if not regions_dict:
            continue

        for reg in (reg_a, reg_b):
            ids = regions_dict.get(reg, np.array([]))
            for uid in ids:
                mask = (
                    (matched_df_pair['source_file'] == session)
                    & (matched_df_pair['region_group'] == reg)
                    & (matched_df_pair['unique_cellid'] == uid)
                )
                auc_val = matched_df_pair.loc[mask, 'auc'].values
                abs_val = matched_df_pair.loc[mask, 'abs_auc_deviation'].values
                fr_val = matched_df_pair.loc[mask, 'firing_rate'].values

                row = {
                    'pair': pair_label,
                    'session': session.replace('_CellsAndTrials.pkl', ''),
                    'region': reg,
                    'unique_cellid': uid,
                    'auc': auc_val[0] if len(auc_val) > 0 else np.nan,
                    'abs_auc_deviation': abs_val[0] if len(abs_val) > 0 else np.nan,
                    'firing_rate': fr_val[0] if len(fr_val) > 0 else np.nan,
                }
                rows.append(row)
                pair_rows.append(row)

    # Save one CSV for just this region pair.
    pair_csv = pd.DataFrame(pair_rows)
    pair_out_path = os.path.join(
        pair_csv_dir,
        f'matched_units_{pair_label.replace("-", "_")}.csv',
    )
    pair_csv.to_csv(pair_out_path, index=False)
    print(f'Saved {len(pair_csv)} rows to {pair_out_path}')

matched_csv = pd.DataFrame(rows)
out_path = 'matched_pairwise_unit_ids_and_auc.csv'
matched_csv.to_csv(out_path, index=False)
print(f'Saved {len(matched_csv)} rows to {out_path}')
print()
if len(matched_csv) > 0:
    print('Matched neurons per pair x region:')
    print(matched_csv.groupby(['pair', 'region']).size().unstack(fill_value=0))
else:
    print('No matched neurons were exported.')


## Summary: MMDDM inclusion criteria + pairwise AUC matching

| Step | This notebook |
|------|---------------|
| **Baseline inclusion** | Keeps only neurons with `inclusion == 1` from the loaded `.pkl` files |
| **Firing rate** | `len(spike_times) / recording_duration >= 2` sp/s |
| **AUC threshold** | `|AUC - 0.5| > 2⁻⁵` (= 0.03125) |
| **Trial filter** | `~violated`, `cpoke_out > cpoke_req_end`, `response_time ≤ 5 s`, `stim_dur ≥ 0.5 s` |
| **Region exclusion** | Excludes `CC` and `unknown` |
| **Region grouping** | `M2` + `Cg1` → `dmFC`; `MO` + `PrL` → `mPFC`; `DGd` + `DGv` + `DG` + `CA1` → `HPC` |
| **Matching strategy** | Match `|AUC - 0.5|` for all 45 pairings among ADS, BLA, dmFC, HPC, M1, MGB, mPFC, NAc, S1, TS |

**Note on FR computation**: If `recording_duration_s` or `fileTimeSecs` is not available in the `.pkl` metadata, the notebook falls back to estimating recording duration from the latest spike time across all neurons. This is a lower bound on `fileTimeSecs` and will slightly overestimate firing rates for neurons that stop firing before the end of the recording.
